# 02 — Prédire l'insatisfaction à l'atterrissage

Responsable : **Jeanne** — branche `modeles`

Données : **train uniquement**, comparaison des modèles en validation croisée.

`test.csv` n'est **jamais** lu ici : voir le notebook 03.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

from src.data import load_train, TARGET, RANDOM_STATE
from src.features import LANDING_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.economics import gain_net, BREAK_EVEN_PRECISION

df = load_train()
X = df[LANDING_FEATURES]
y = df[TARGET]
X.shape, y.mean()

((103904, 7), np.float64(0.5666673082845703))

## 1. Variables connues à l'atterrissage
Justification de chaque variable retenue ou exclue (voir `src/features.py`).

## 2. Méthode de comparaison
Validation croisée stratifiée à 5 plis, `random_state=42`, pipeline (encodage + imputation du retard d'arrivée).

## 3. Règle simple sans modèle (baseline)
Hasard (10 % des passagers) et règle métier.

In [2]:
import numpy as np

rng = np.random.default_rng(RANDOM_STATE)   # hasard reproductible : même tirage à chaque exécution
scores_hasard = rng.random(len(y))          # un score aléatoire par passager

gain_net(y, scores_hasard)

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 5876,
 'precision': 0.5655437921077959,
 'rappel': 0.0997978905891744,
 'gain_net': 172000.0,
 'gain_net_pour_1000': 1655.3741915614412}

In [6]:
# Retard d'arrivée manquant (310 passagers) : on le remplace par le retard au départ,
# les deux sont quasi identiques (corrélation 0,97)
retard = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])

# Règle 1 : aborder les plus gros retards à l'arrivée
regle_retard = gain_net(y, retard)

# Règle 2 : voyage perso d'abord, puis les plus gros retards
perso = (X["Type of Travel"] == "Personal Travel").astype(int)
regle_perso = gain_net(y, perso * 10_000 + retard)

import pandas as pd
baselines = pd.DataFrame(
    [gain_net(y, scores_hasard), regle_retard, regle_perso],
    index=["Hasard", "Règle : plus gros retards", "Règle : voyage perso puis retard"],
)
baselines[["precision", "rappel", "gain_net_pour_1000"]].round(3)

,precision,rappel,gain_net_pour_1000
Hasard,0.566,0.100,1655.374
Règle : plus gros retards,0.642,0.113,2422.428
Règle : voyage perso puis retard,0.956,0.169,5564.752


## 4. Modèle 1 — régression logistique

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Mêmes données que les règles : retard d'arrivée manquant = retard au départ
X_modele = X.copy()
X_modele["Arrival Delay in Minutes"] = retard

# Préparation : nombres mis à la même échelle, catégories transformées en colonnes 0/1
preparation = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
logistique = make_pipeline(preparation, LogisticRegression(max_iter=1000))

# Validation croisée en 5 parts, toujours le même découpage (random_state)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_logistique = cross_val_predict(
    logistique, X_modele, y, cv=cv, method="predict_proba"
)[:, 1]

resultat_logistique = gain_net(y, scores_logistique)
resultat_logistique

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 9732,
 'precision': 0.936669874879692,
 'rappel': 0.16528813329030723,
 'gain_net': 557600.0,
 'gain_net_pour_1000': 5366.492146596858}

## 5. Modèle 2 — forêt aléatoire / gradient boosting

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Les arbres n'ont pas besoin de mise à l'échelle : les nombres passent tels quels
preparation_arbres = ColumnTransformer([
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
boosting = make_pipeline(
    preparation_arbres,
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)

# Même découpage en 5 parts que la logistique : la comparaison est équitable
scores_boosting = cross_val_predict(
    boosting, X_modele, y, cv=cv, method="predict_proba"
)[:, 1]

resultat_boosting = gain_net(y, scores_boosting)
resultat_boosting

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


{'passagers_abordes': 10390,
 'insatisfaits_abordes': 10049,
 'precision': 0.9671799807507219,
 'rappel': 0.17067205625095536,
 'gain_net': 589300.0,
 'gain_net_pour_1000': 5671.581459809055}

## 6. Comparaison et choix du modèle
Précision dans le top 10 %, gain net pour 1 000 passagers, coût des erreurs, lisibilité.

In [9]:
maximum = gain_net(y, y.values)["gain_net_pour_1000"]   # si on n'abordait que des insatisfaits

comparaison = pd.concat([
    baselines,
    pd.DataFrame(
        [resultat_logistique, resultat_boosting],
        index=["Régression logistique", "Gradient boosting"],
    ),
])
comparaison["% du gain maximal"] = comparaison["gain_net_pour_1000"] / maximum * 100

comparaison[["precision", "rappel", "gain_net_pour_1000", "% du gain maximal"]].round(3)

,precision,rappel,gain_net_pour_1000,% du gain maximal
Hasard,0.566,0.100,1655.374,27.591
Règle : plus gros retards,0.642,0.113,2422.428,40.375
Règle : voyage perso puis retard,0.956,0.169,5564.752,92.749
Régression logistique,0.937,0.165,5366.492,89.445
Gradient boosting,0.967,0.171,5671.581,94.530
